# 02 · Descritores DINOv2 a partir de uma spec

Renderiza os recortes da spec na GPU (`dal.v3.sensor.render`), extrai DINOv2 (GeM, CLS, grade ou a saída `ft` do ajuste fino, conforme `outputs`) e, se a config pedir, o hand da v1 nos mesmos pixels. Grava `float16 [N, D]` em `MyDrive/drone_v3/<output_dir>/` com `MANIFEST.sha256`, e o resumo em `colab_out/<job_id>/`.

Antes: rode `00_setup.ipynb`. O job é **retomável**: se a sessão cair, rode o setup e este notebook de novo com o mesmo `JOB_ID`.

In [ ]:
JOB_ID = 'g1_legacy_dinov2'
import pandas as pd
jobs = pd.read_csv('/content/repo/colab/jobs_v3.csv')
job = jobs.set_index('job_id').loc[JOB_ID]
JOB_CONFIG = job.config_path
print(job)

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'dal.v3.descriptors.extract', '--job-config', JOB_CONFIG, '--job-id', JOB_ID,
       '--out-root', '/content/drive/MyDrive/drone_v3', '--colab-out', 'colab_out']
print(subprocess.run(cmd, cwd='/content/repo', check=True).returncode)

In [ ]:
# G1: recuperação com a mesma função do REPORT.md §2 (window_stats), no próprio Colab.
# g1_splits: faixas das janelas (o ajuste fino usa a de teste); g1_compare: descritores já no Drive, nas mesmas janelas.
import json, yaml, pathlib
cfg = yaml.safe_load(open(f'/content/repo/{JOB_CONFIG}'))
drive = '/content/drive/MyDrive/drone_v3'
base = f"{drive}/{cfg['output_dir']}"
files = [f for f in sorted(pathlib.Path(base).glob('*.npy')) if f.stem not in ('valid_frac', 'cloud_frac')]
files += [pathlib.Path(drive) / extra for extra in cfg.get('g1_compare', [])]
for split in cfg.get('g1_splits', ['train']):
    for path in files:
        name = path.stem if path.parent == pathlib.Path(base) else f'{path.parent.name}_{path.stem}'
        subprocess.run([sys.executable, 'scripts/eval_retrieval_v3.py', '--spec', cfg['spec'], '--descriptors',
                        str(path), '--name', f'{JOB_ID}_{name}', '--split', split, '--out', f'colab_out/{JOB_ID}'],
                       cwd='/content/repo', check=True)

In [ ]:
import json
done = json.load(open(f'/content/repo/colab_out/{JOB_ID}/DONE.json'))
print(json.dumps(done, indent=1))
main_number = str(done['metrics'].get('crops_per_second', ''))[:12]
subprocess.run([sys.executable, 'scripts/colab_publish_v3.py', '--job-id', JOB_ID,
                '--main-number', f'crops/s {main_number}'], cwd='/content/repo', check=True)